# 集成学习第三课：随机森林复盘

## 本课目标

把随机森林放回集成学习框架里理解：它和普通 Bagging 的关系是什么，为什么它通常比单棵决策树更稳。

一句话版：

```text
随机森林 = Bagging + 决策树 + 特征随机
```

## 1. 从 Bagging 到随机森林

上一课学习了 Bagging：

```text
Bagging = Bootstrap 抽样 + 多个基学习器 + 投票 / 平均
```

如果基学习器使用决策树，就得到一种很自然的模型组合：

```text
多份 Bootstrap 数据 -> 多棵决策树 -> 投票 / 平均
```

随机森林在这个基础上，又增加了一个关键随机性：

```text
每个节点分裂时，不看全部特征，只随机看一部分特征。
```

## 2. 为什么还要随机选特征？

普通 Bagging 已经让每棵树看到不同样本了，为什么随机森林还要随机选特征？

原因是：如果数据里有一个特别强的特征，很多树可能都会优先使用它分裂。

这样会导致：

```text
树和树之间仍然很像，集成效果就会打折。
```

随机选特征的目的，是让树之间更不一样。

也就是：

```text
样本随机 -> 每棵树看到的数据不同
特征随机 -> 每棵树在分裂时考虑的视角不同
```

## 3. 三个模型的关系

| 模型 | 样本随机 | 特征随机 | 模型数量 |
|---|---|---|---|
| 单棵决策树 | 否 | 否 | 1 |
| Bagging 决策树 | 是 | 通常否 | 多棵 |
| 随机森林 | 是 | 是 | 多棵 |

可以这样记：

```text
单棵树：一个人做判断。
Bagging：很多人看不同题本后投票。
随机森林：很多人不但看不同题本，而且每次答题关注的特征也不完全一样。
```

## 4. sklearn 实战：三者对比

下面用 `make_moons` 构造一个非线性分类数据集，对比：

- 单棵决策树
- BaggingClassifier + 决策树
- RandomForestClassifier

In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score

X, y = make_moons(n_samples=600, noise=0.3, random_state=22)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=22, stratify=y
)

tree = DecisionTreeClassifier(random_state=22)
tree.fit(X_train, y_train)

try:
    bagging = BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=22),
        n_estimators=100,
        bootstrap=True,
        random_state=22,
        n_jobs=-1,
    )
except TypeError:
    bagging = BaggingClassifier(
        base_estimator=DecisionTreeClassifier(random_state=22),
        n_estimators=100,
        bootstrap=True,
        random_state=22,
        n_jobs=-1,
    )

bagging.fit(X_train, y_train)

forest = RandomForestClassifier(
    n_estimators=100,
    random_state=22,
    n_jobs=-1,
)
forest.fit(X_train, y_train)

scores = {
    "单棵决策树": accuracy_score(y_test, tree.predict(X_test)),
    "Bagging 决策树": accuracy_score(y_test, bagging.predict(X_test)),
    "随机森林": accuracy_score(y_test, forest.predict(X_test)),
}

scores

一次实验的分数不一定每次都严格满足“随机森林最高”，但通常会看到：

```text
集成模型比单棵树更稳定。
```

随机森林的优势尤其体现在：数据维度更高、特征之间有冗余、单棵树容易过拟合的时候。

## 5. 多次划分：看稳定性

继续用多次随机划分来观察稳定性。

重点看：

- 平均准确率。
- 准确率标准差。

标准差越小，说明模型对数据划分越不敏感。

In [ ]:
tree_scores = []
bagging_scores = []
forest_scores = []

for seed in range(30):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=seed, stratify=y
    )

    tree = DecisionTreeClassifier(random_state=seed)
    tree.fit(X_train, y_train)
    tree_scores.append(accuracy_score(y_test, tree.predict(X_test)))

    try:
        bagging = BaggingClassifier(
            estimator=DecisionTreeClassifier(random_state=seed),
            n_estimators=100,
            bootstrap=True,
            random_state=seed,
            n_jobs=-1,
        )
    except TypeError:
        bagging = BaggingClassifier(
            base_estimator=DecisionTreeClassifier(random_state=seed),
            n_estimators=100,
            bootstrap=True,
            random_state=seed,
            n_jobs=-1,
        )
    bagging.fit(X_train, y_train)
    bagging_scores.append(accuracy_score(y_test, bagging.predict(X_test)))

    forest = RandomForestClassifier(
        n_estimators=100,
        random_state=seed,
        n_jobs=-1,
    )
    forest.fit(X_train, y_train)
    forest_scores.append(accuracy_score(y_test, forest.predict(X_test)))

summary = {
    "单棵树平均准确率": np.mean(tree_scores),
    "单棵树标准差": np.std(tree_scores),
    "Bagging 平均准确率": np.mean(bagging_scores),
    "Bagging 标准差": np.std(bagging_scores),
    "随机森林平均准确率": np.mean(forest_scores),
    "随机森林标准差": np.std(forest_scores),
}

summary

## 6. 随机森林的重要参数

`RandomForestClassifier` 常见参数：

| 参数 | 含义 |
|---|---|
| `n_estimators` | 树的数量 |
| `max_depth` | 每棵树的最大深度 |
| `max_features` | 每次分裂时最多考虑多少特征 |
| `min_samples_split` | 一个节点至少有多少样本才继续分裂 |
| `min_samples_leaf` | 叶子节点至少包含多少样本 |
| `bootstrap` | 是否使用有放回抽样 |
| `oob_score` | 是否使用袋外样本评估 |
| `random_state` | 随机种子 |
| `n_jobs` | 并行训练使用的 CPU 数量 |

学习初期最常关注：

- `n_estimators`
- `max_depth`
- `max_features`
- `oob_score`

## 7. `max_features`：特征随机的开关

`max_features` 控制每个节点分裂时考虑多少特征。

例如有 20 个特征：

- `max_features=20`：每次分裂都看全部特征，树之间更像。
- `max_features="sqrt"`：每次大约看 sqrt(20) 个特征，树之间差异更大。
- `max_features=0.5`：每次看一半特征。

在分类任务中，sklearn 随机森林默认常用 `sqrt` 思路。

In [ ]:
# 在这个二维数据集上，max_features 的差异不明显。
# 这里主要看写法：
forest_sqrt = RandomForestClassifier(
    n_estimators=100,
    max_features="sqrt",
    random_state=22,
    n_jobs=-1,
)

forest_all = RandomForestClassifier(
    n_estimators=100,
    max_features=None,
    random_state=22,
    n_jobs=-1,
)

forest_sqrt.fit(X_train, y_train)
forest_all.fit(X_train, y_train)

accuracy_score(y_test, forest_sqrt.predict(X_test)), accuracy_score(y_test, forest_all.predict(X_test))

## 8. OOB：随机森林的袋外评估

随机森林默认使用 Bootstrap 抽样，所以每棵树都有一部分训练样本没有见过。

这些没被某棵树抽到的样本可以用于 OOB 评估。

写法：

```python
RandomForestClassifier(oob_score=True)
```

训练后查看：

```python
model.oob_score_
```

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=22, stratify=y
)

forest_oob = RandomForestClassifier(
    n_estimators=200,
    oob_score=True,
    random_state=22,
    n_jobs=-1,
)
forest_oob.fit(X_train, y_train)

oob_score = forest_oob.oob_score_
test_score = accuracy_score(y_test, forest_oob.predict(X_test))

oob_score, test_score

OOB 分数可以理解为一种内部验证分数。

它不能完全替代测试集，但在调参时很有参考价值。

## 9. 特征重要性

随机森林可以给出每个特征的重要性。

在 sklearn 中查看：

```python
model.feature_importances_
```

下面用乳腺癌分类数据集演示。这个数据集特征更多，比二维月牙数据更适合看特征重要性。

In [ ]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()
X_cancer = cancer.data
y_cancer = cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X_cancer, y_cancer, test_size=0.3, random_state=22, stratify=y_cancer
)

forest_cancer = RandomForestClassifier(
    n_estimators=200,
    random_state=22,
    n_jobs=-1,
)
forest_cancer.fit(X_train, y_train)

importances = forest_cancer.feature_importances_
top_indices = np.argsort(importances)[::-1][:8]

top_features = [
    (cancer.feature_names[i], importances[i])
    for i in top_indices
]

accuracy_score(y_test, forest_cancer.predict(X_test)), top_features

特征重要性可以帮助我们理解模型主要依赖哪些特征。

但要注意：

- 它不是因果解释。
- 特征之间强相关时，重要性可能被分散。
- 它适合做初步分析，不适合直接当作绝对结论。

## 10. 随机森林的优缺点

优点：

- 通常比单棵决策树更准确、更稳定。
- 对特征缩放不敏感。
- 能处理分类和回归任务。
- 可以输出特征重要性。
- 参数相对容易上手。

缺点：

- 可解释性不如单棵决策树。
- 训练和预测比单棵树更慢。
- 模型体积更大。
- 对特别稀疏、高维的数据，不一定总是最佳选择。

## 11. 本课小结

- 随机森林是 Bagging 思想在决策树上的重要扩展。
- Bagging 主要做样本随机。
- 随机森林额外加入特征随机。
- 样本随机和特征随机都是为了让树之间更有差异。
- 多棵差异化的树投票或平均，能降低方差，让模型更稳定。
- 随机森林支持 OOB 评估和特征重要性分析。

记忆方式：

```text
随机森林 = 很多棵不完全一样的树，一起投票。
不一样来自两处：样本随机 + 特征随机。
```